In [19]:
%load_ext ROOT._jupyroot.magics.jsrootmagic
import sys
print(sys.executable)
import ROOT
print(ROOT.gROOT.GetVersion())

The ROOT._jupyroot.magics.jsrootmagic extension is already loaded. To reload it, use:
  %reload_ext ROOT._jupyroot.magics.jsrootmagic
/home/tobiaskabelac/Documents/GitHub/Maturitni-prace/.venv/bin/python
6.40.00


In [20]:
# Cell 1: Imports and setup
# Earlier version used: import ROOT, ROOT.gStyle.SetOptFit(1111), math, os, pandas
# ROOT is used for histogramming and fitting; the style setting makes the fit output appear in the usual ROOT format.
import ROOT
ROOT.gStyle.SetOptFit(1111)
import math
import os
import pandas


In [21]:
# Cell 2: Load data and create ROOT tree
# Earlier version used: csv_file = "data/Zmumu.csv"
# The CSV is loaded into a ROOT tree and then saved as a temporary ROOT file for later access.
csv_file = "data/Zmumu.csv"

data = ROOT.RDF.FromCSV(csv_file)
data.Snapshot('my_tree', 'my_data.root')

my_tree = ROOT.TChain("my_tree")
my_tree.Add("my_data.root")

# Optional check
my_tree.GetEntries()  # returns the number of events in the tree


10000

In [13]:
# Cell 3: Define histogram
# Earlier version used: x_low = 70, x_up = 110, n_bins = 200
# This defines the mass range for the reconstructed dimuon invariant-mass spectrum.
x_low   = 70   # lower limit of the histogram in GeV
x_up    = 110  # upper limit of the histogram in GeV
n_bins  = 200  # number of bins
x_title = "M_{#mu#mu} [GeV/c^{2}]"  # x-axis label
y_title = "Events"  # y-axis label
title   = "Dimuon mass"  # histogram title

hist = ROOT.TH1F("hist", title + ";" + x_title + ";" + y_title, n_bins, x_low, x_up)


Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).


In [14]:
# Cell 4: Fill invariant-mass histogram
# Earlier version used the same logic, but this block was kept separate to make debugging easier.
# It reconstructs each muon as a TLorentzVector and fills the histogram with the invariant mass dimuon.M().
import math  # needed for the final signal estimate

muon_mass = 0.10566  # muon mass in GeV/c^2
hist.Reset()  # clear the histogram if the cell is executed again

for i in range(my_tree.GetEntries()):
    my_tree.GetEntry(i)  # read the i-th event from the tree

    muon1 = ROOT.TLorentzVector()  # four-vector for the first muon
    muon1.SetPtEtaPhiM(my_tree.pt1, my_tree.eta1, my_tree.phi1, muon_mass)

    muon2 = ROOT.TLorentzVector()  # four-vector for the second muon
    muon2.SetPtEtaPhiM(my_tree.pt2, my_tree.eta2, my_tree.phi2, muon_mass)

    dimuon = muon1 + muon2  # add the two four-vectors to get the dimuon system
    hist.Fill(dimuon.M())  # invariant mass M = sqrt((p1 + p2)^2)


In [15]:
# Cell 5: Draw the dimuon mass spectrum
# Interactive JSROOT plots work in JupyterLab; use %jsroot off for static PNGs in VS Code.
%jsroot on
can = ROOT.TCanvas()  # create the canvas
can.Draw()  # display the canvas
hist.Draw("E1")  # draw the histogram with error bars


In [16]:
# Cell 6: First rough fit
# A Gaussian + linear background model is robust for the Z peak and gives a stable
# estimate of the mass and width.
x_low_fit = x_low  # lower fit bound
x_up_fit = x_up    # upper fit bound

my_fit = ROOT.TF1("my_fit", "gaus(0)+pol1(3)", x_low_fit, x_up_fit)
my_fit.SetParNames("A", "Mu", "Sigma", "c", "b")
my_fit.SetParameters(200, 91, 5, 0, 0)
hist.Fit("my_fit", "R")
can.Draw()
hist.Draw("E")


****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      469.292
NDf                       =          195
Edm                       =  1.19009e-06
NCalls                    =          173
A                         =      238.448   +/-   4.26558     
Mu                        =      90.7805   +/-   0.0304606   
Sigma                     =        2.266   +/-   0.032737    
c                         =       41.984   +/-   1.69583     
b                         =    -0.358116   +/-   0.0170788   


In [17]:
# Cell 7: Final fit of the Z peak with background
# Earlier version used:
# my_fit_pol1 = ROOT.TF1("my_fit_pol1", "gaus(0)+pol1(3)", x_low_fit, x_up_fit)
# my_fit_pol1.SetParNames("A", "Mu", "Sigma", "c", "b")
# my_fit_pol1.SetParameters(200, 91, 5, 0, 0)
# However, the actual final model used here is a CrystalBall + linear background.
# my_fit_pol1 = ROOT.TF1("my_fit_pol1", "crystalball + pol1(6)", x_low_fit, x_up_fit)  # signal model + background
# my_fit_pol1.SetParNames("A", "Mu", "Sigma", "alpha", "n", "c", "b")  # names for the CrystalBall and linear terms
# my_fit_pol1.SetParameters(200, 91, 5, 2, 0.5, 0, 0)  # starting values for all parameters
# my_fit_pol1.SetMaxIterations(100)  # limit the number of fitting iterations

# hist.Fit("my_fit_pol1", "REM", "", x_low_fit, x_up_fit)  # R = fit range, E = error bars, M = improve fit output
# can.Draw()
# hist.Draw("E")

# mu = my_fit_pol1.GetParameter("Mu")  # extracted peak position (mass)
# mu_error = my_fit_pol1.GetParError(1)  # uncertainty of the mass parameter
# print("Z boson mass: ", round(mu, 4), " +- ", round(mu_error, 4), " GeV/c^2")


In [18]:
# Cell 8: Save plot and print signal estimate
# and a simple estimate of the signal based on the peak shape and bin width.

signal = (math.sqrt(2 * math.pi) * 252 * 2.102) / ((x_up - x_low) / n_bins)  # rough signal estimate
print(signal)


6638.8551179256965


# Maturitni-prace